# 환경설정

In [ ]:
# 인증

from google.colab import auth
auth.authenticate_user()
print("인증 완료")

In [ ]:
# 매직 로드

%load_ext google.cloud.bigquery

# stage.stg_group

**목적**: `accounts_group`를 분석용 Stage로 정제
**산출물**: `stage.stg_group` (VIEW)

**핵심 정보 요약**
- invalid_class_num_count (2건)-> 클래스 넘버가 '0' 으로 입력
- invalid_grade_range (1건) -> max_grade가 20
- invalid_class_num_range (3건) -> min_class_num이 0, max_class_num이 41까지 확인
- 이상치 4개 확인
- accounts_group 테이블 내 school_id = 1을 참조하고 있는 26개의 학급 데이터가 존재하나, 마스터 테이블인 accounts_school에는 id = 1에 해당하는 학교 정보가 누락
    - 영향: 두 테이블 간 INNER JOIN 시 해당 26개 행이 누락(고아 데이터)되는 현상 발생.

    - 조치 방침: Staging 단계에서는 원본 무결성을 유지하기 위해 수정하지 않으며, 향후 Mart(마트) 단계 또는 EDA 분석 시 목적에 맞게 처리(LEFT JOIN 유지 또는 필터링) 필요.

**Raw 품질**: 중복 0 · 결측 0 · status 예상 외 값 2 · PK(id) 유니크

## Raw 품질 확인

### 데이터 샘플 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT *
FROM `your-gcp-project.raw.accounts_group`
LIMIT 5;

### 1. 컬럼 타입 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    column_name,
    data_type
FROM `your-gcp-project.raw.INFORMATION_SCHEMA.COLUMNS`
WHERE table_name = 'accounts_group';

### 2. 중복, 결측치, 이상치 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_id_count,
    -- 결측치(NULL) 검증
    COUNTIF(id IS NULL) AS null_id,
    COUNTIF(school_id IS NULL) AS null_school_id,
    COUNTIF(grade IS NULL) AS null_grade,
    COUNTIF(class_num IS NULL) AS null_class_num,
    -- 비정상 값 검증 (예: 학년이나 반이 0 이하이거나 터무니없이 큰 경우 등, 필요시 도메인 조건 추가)
    COUNTIF(grade <= 0) AS invalid_grade_count,
    COUNTIF(class_num <= 0) AS invalid_class_num_count
FROM `your-gcp-project.raw.accounts_group`;

#### invalid_class_num_count (2건) 조사

In [ ]:
%%bigquery --project your-gcp-project
SELECT *
FROM `your-gcp-project.raw.accounts_group`
WHERE class_num <= 0 OR grade <= 0;

### 3. grade & class_num 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    -- 학년이나 반이 학교 급별 통상적인 범위를 벗어나는지 확인 (예: 3학년 초과 혹은 0 이하 등)
    COUNTIF(grade <= 0 OR grade > 4) AS invalid_grade_range,
    COUNTIF(class_num <= 0 OR class_num > 20) AS invalid_class_num_range,
    -- 통계를 위한 최솟값, 최댓값, 평균값 확인
    MIN(grade) AS min_grade,
    MAX(grade) AS max_grade,
    MIN(class_num) AS min_class_num,
    MAX(class_num) AS max_class_num
FROM `your-gcp-project.raw.accounts_group`;

#### 전체 이상치 확인

In [ ]:
%%bigquery --project your-gcp-project
SELECT *
FROM `your-gcp-project.raw.accounts_group`
WHERE grade <= 0 OR grade > 4 OR class_num <= 0 OR class_num > 20;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    COUNT(*) AS total_outlier_rows
FROM `your-gcp-project.raw.accounts_group`
WHERE grade <= 0 OR grade > 4 OR class_num <= 0 OR class_num > 20;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    'grade' AS column_name,
    MIN(grade) AS min_val,
    APPROX_QUANTILES(grade, 100)[OFFSET(25)] AS q25,
    APPROX_QUANTILES(grade, 100)[OFFSET(50)] AS median,
    APPROX_QUANTILES(grade, 100)[OFFSET(75)] AS q75,
    MAX(grade) AS max_val
FROM `your-gcp-project.raw.accounts_group`
UNION ALL
SELECT
    'class_num' AS column_name,
    MIN(class_num) AS min_val,
    APPROX_QUANTILES(class_num, 100)[OFFSET(25)] AS q25,
    APPROX_QUANTILES(class_num, 100)[OFFSET(50)] AS median,
    APPROX_QUANTILES(class_num, 100)[OFFSET(75)] AS q75,
    MAX(class_num) AS max_val
FROM `your-gcp-project.raw.accounts_group`;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    CASE
        WHEN grade BETWEEN 1 AND 3 THEN '정상 (1~3학년)'
        ELSE '이상치 (학년 범위 외)'
    END AS grade_status,
    CASE
        WHEN class_num BETWEEN 1 AND 20 THEN '정상 (1~20반)'
        ELSE '이상치 (반 범위 외)'
    END AS class_status,
    COUNT(*) AS row_count
FROM `your-gcp-project.raw.accounts_group`
GROUP BY grade_status, class_status
ORDER BY grade_status, class_status;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    id,
    grade,
    class_num,
    school_id,
    CASE
        WHEN grade <= 0 OR grade > 4 THEN '학년 이상치'
        WHEN class_num <= 0 OR class_num > 20 THEN '반 이상치'
        ELSE '정상'
    END AS outlier_type
FROM `your-gcp-project.raw.accounts_group`
WHERE grade <= 0 OR grade > 4 OR class_num <= 0 OR class_num > 20;

### 4. school_id 매칭 확인(생략)

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    COUNTIF(g.school_id NOT IN (SELECT id FROM `your-gcp-project.raw.accounts_school`)) AS orphan_school_id_count
FROM `your-gcp-project.raw.accounts_group` g;

#### 26개 탐구

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    g.id AS group_id,
    g.grade,
    g.class_num,
    g.school_id AS group_school_id,
    s.id AS school_table_id,
    s.address,
    s.school_type,
    CASE
        WHEN s.id IS NULL THEN '고아 데이터 (미매칭)'
        ELSE '정상 매칭'
    END AS match_status
FROM `your-gcp-project.raw.accounts_group` g
LEFT JOIN `your-gcp-project.raw.accounts_school` s
    ON g.school_id = s.id
ORDER BY match_status DESC, g.id;

In [ ]:
%%bigquery --project your-gcp-project
SELECT
    g.id AS group_id,
    g.grade,
    g.class_num,
    g.school_id
FROM `your-gcp-project.raw.accounts_group` g
WHERE g.school_id NOT IN (
    SELECT id FROM `your-gcp-project.raw.accounts_school`
)
ORDER BY g.id;

In [ ]:
%%bigquery --project your-gcp-project
SELECT *
FROM `your-gcp-project.raw.accounts_school`
WHERE id = 1;

## 최종 Stage

### view (관례)

In [ ]:
%%bigquery --project your-gcp-project
CREATE OR REPLACE VIEW `your-gcp-project.stage.stg_group` AS
WITH src AS (
  SELECT
    id,
    grade,
    class_num,
    school_id
  FROM `your-gcp-project.raw.accounts_group`
)
SELECT
  id AS group_id,
  grade,
  class_num,
  school_id
FROM src;

### 생성 후 검증

In [ ]:
%%bigquery --project your-gcp-project
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT group_id) AS distinct_group_id,
  COUNTIF(group_id IS NULL) AS null_group_id,
  COUNTIF(school_id IS NULL) AS null_school_id,
  COUNTIF(grade <= 0 OR grade > 4) AS invalid_grade_count,
  COUNTIF(class_num <= 0 OR class_num > 20) AS invalid_class_num_count
FROM `your-gcp-project.stage.stg_group`;